# Atelier 2.C — Évaluation RAGAS et reranking ★

**Durée : 70 min** · Projet fil rouge, incrément du Jour 2

Un assistant RAG peut échouer à deux endroits : la **recherche** (les bons extraits ne sont pas retrouvés) et la **génération** (les extraits sont là, mais la réponse est fausse, infidèle ou hors sujet). Cet atelier mesure les deux, sur un **golden set** de questions et de réponses de référence :

1. métriques de recherche sans LLM : *context recall* et *context precision@k*, et apport du reranker ;
2. métriques RAGAS calculées par un LLM juge : *faithfulness*, *answer relevancy*, *context precision*, *context recall*, *answer correctness* ;
3. diagnostic des erreurs, puis enregistrement du **score de référence** ;
4. observabilité et expériences avec **LangSmith** (facultatif).

Livrable : `resultats/score_reference.json`.

In [1]:
import json
import os
from datetime import date
from pathlib import Path

import pandas as pd

import evaluation_rag as ev
import outils_llm as llm
import rag

MODELE = llm.choisir_modele()   # modèle de l'assistant
MODELE_JUGE = os.getenv("MODELE_JUGE") or MODELE   # idéalement un autre modèle, au moins aussi capable
N_MAX = 12                      # questions évaluées par RAGAS (environ 10 appels au juge chacune) ; None pour toutes
K = 4                           # extraits transmis au modèle
print("Assistant :", MODELE, "| juge :", MODELE_JUGE)
print("LangSmith :", "traces actives" if ev.activer_langsmith() else "non configuré")

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Assistant : groq:openai/gpt-oss-20b | juge : nvidia:nvidia/nemotron-3.5-lightning-30b-a3b
LangSmith : non configuré


## 1. Le golden set

Le golden set (`donnees/golden_set.json`) est le jeu de référence de l'assistant. Chaque entrée contient :

| Champ | Rôle |
|---|---|
| `question` | Formulée comme un vrai client, pas comme le document |
| `reponse_reference` | Réponse vérifiée par un expert métier |
| `contextes_reference` | Passages exacts du corpus qui justifient la réponse |
| `sources` | Documents d'où viennent ces passages |
| `indices` | Courts passages indispensables, pour mesurer la recherche sans LLM |
| `type` | factuelle, tableau, calcul, multi-sources, hors corpus, interne |

Un bon golden set couvre tous les documents et tous les types de questions, y compris des questions **hors corpus** (réponse attendue : « je ne trouve pas ») et des questions **sensibles** (documents internes). Il est versionné et enrichi avec les questions réelles qui posent problème.

In [2]:
golden = rag.charger_golden_set()
print(len(golden), "entrées")
display(pd.DataFrame(golden)[["id", "type", "question", "reponse_reference"]].head(8))
pd.Series([g["type"] for g in golden]).value_counts()

32 entrées


,id,type,question,reponse_reference
0,Q01,factuelle,Quel est le montant minimum d'une commande ?,100 CHF hors taxes.
1,Q02,factuelle,À partir de quel montant la livraison est-elle...,Dès 500 CHF hors taxes de commande ; en dessou...
2,Q03,factuelle,Quel est le délai de paiement standard des fac...,"30 jours net à compter de la date de facture, ..."
3,Q04,factuelle,Que se passe-t-il si je paie une facture en re...,"Intérêts moratoires de 5 % par an, frais de ra..."
4,Q05,factuelle,Accordez-vous une remise sur les grosses comma...,"Oui, 10 % sur toute commande supérieure à 2 00..."
5,Q06,multi-sources,Une licence logicielle peut-elle être rembours...,"Non, une licence activée n'est ni reprise ni r..."
6,Q07,factuelle,Quel tribunal est compétent en cas de litige a...,Le for juridique est à Lausanne ; le droit sui...
7,Q08,factuelle,De combien de temps dispose un client Premium ...,60 jours après la livraison.


factuelle        17
tableau           6
multi-sources     4
calcul            2
hors_corpus       2
interne           1
Name: count, dtype: int64

In [3]:
print(json.dumps(next(g for g in golden if g["id"] == "Q18"), ensure_ascii=False, indent=2))

{
  "id": "Q18",
  "type": "tableau",
  "question": "Quel est le délai d'intervention du contrat de maintenance Argent ?",
  "reponse_reference": "8 heures ouvrées.",
  "contextes_reference": [
    "Argent : délai d'intervention de 8 heures ouvrées, du lundi au vendredi de 8 h à 17 h, pour 12 % de la valeur du matériel couvert."
  ],
  "sources": [
    "conditions_maintenance.pdf"
  ],
  "indices": [
    "8 heures ouvrées"
  ]
}


## 2. Évaluer la recherche, sans LLM

Pour chaque question, on regarde les `K` extraits retrouvés :

- **context recall** : part des passages indispensables (`indices`) présents dans les extraits ;
- **precision@k** : part des `K` extraits qui sont pertinents (qui contiennent au moins un indice) ;
- **context precision@k** : moyenne des precision@i aux rangs où se trouve un extrait pertinent. Elle vaut 1 quand les extraits pertinents sont tous **en tête**, et baisse quand ils arrivent tard ;
- **reciprocal rank** : 1 / rang du premier extrait pertinent.

Exemple avec K = 4 : extraits pertinents aux rangs 1 et 3. precision@4 = 2/4 = 0,5 ; context precision@4 = (1/1 + 2/3) / 2 ≈ 0,83.

Ces mesures sont gratuites et reproductibles : on peut comparer des dizaines de configurations.

In [4]:
documents = rag.charger_corpus()
encodeur = rag.Encodeur()
index = rag.Index(rag.decouper(documents, "sections", 800, 100, entete=True), encodeur)
reranker = rag.Reranker()   # premier lancement : téléchargement du modèle (0,57 Go)
avec_contexte = [g for g in golden if g["indices"]]

def evaluer_recherche(rechercher):
    lignes = [ev.metriques_recherche(rechercher(g["question"]), g) for g in avec_contexte]
    return pd.DataFrame(lignes).mean().round(3)

configurations = {
    "dense": lambda q: index.dense(q, K),
    "hybride": lambda q: index.hybride(q, K),
    "hybride (20) + reranker": lambda q: reranker.reordonner(q, index.hybride(q, 20), K),
}
recherche = pd.DataFrame({nom: evaluer_recherche(f) for nom, f in configurations.items()}).T
recherche

,context_recall,precision_a_k,context_precision,reciprocal_rank
dense,0.900,0.283,0.767,0.786
hybride,0.900,0.292,0.700,0.717
hybride (20) + reranker,0.967,0.333,0.914,0.925


**À observer.** Le reranker relit 20 candidats avec un cross-encoder et garde les 4 meilleurs : il améliore surtout la **context precision@k** et le reciprocal rank, c'est-à-dire l'ordre des extraits. Le rappel ne peut pas dépasser ce que la recherche hybride a trouvé dans ses 20 candidats.

## 3. Générer les réponses

L'assistant complet : recherche hybride, reranking, génération citée. L'échantillon contient toujours les questions hors corpus et la question sensible.

In [5]:
assistant = rag.Assistant(index, reranker, modele=MODELE, k=K)

from itertools import chain, zip_longest

speciales = [g for g in golden if g["type"] in ("hors_corpus", "interne")]
par_type = {}
for g in golden:
    if g not in speciales:
        par_type.setdefault(g["type"], []).append(g)
# une question de chaque type à tour de rôle, pour que l'échantillon couvre tous les types
ordinaires = [g for g in chain.from_iterable(zip_longest(*par_type.values())) if g]
echantillon = (ordinaires if N_MAX is None else ordinaires[: N_MAX - len(speciales)]) + speciales
print(", ".join(f"{g['id']} ({g['type']})" for g in echantillon))

reponses = llm.en_lot(lambda g: assistant.repondre(g["question"]), echantillon, paralleles=2)

Q01 (factuelle), Q06 (multi-sources), Q11 (tableau), Q17 (calcul), Q02 (factuelle), Q10 (multi-sources), Q14 (tableau), Q19 (calcul), Q03 (factuelle), Q30 (hors_corpus), Q31 (hors_corpus), Q32 (interne)
12 appels terminés, 0 erreur(s).


## 4. Les métriques RAGAS

| Métrique | Question posée | Calcul dans RAGAS | Entrées |
|---|---|---|---|
| **faithfulness** | La réponse dit-elle seulement ce que disent les extraits ? | Le juge découpe la réponse en affirmations et vérifie chacune dans les extraits : affirmations soutenues / total | question, réponse, extraits |
| **answer_relevancy** | La réponse répond-elle à la question posée ? | Le juge génère 3 questions à partir de la réponse ; moyenne de leur similarité cosinus avec la question réelle | question, réponse, embeddings |
| **context_precision** | Les extraits utiles sont-ils en tête ? | Le juge note chaque extrait (utile ou non pour la référence) ; précision moyenne pondérée par le rang | question, référence, extraits |
| **context_recall** | Les extraits contiennent-ils tout ce qu'il faut ? | Affirmations de la référence retrouvées dans les extraits / total | question, référence, extraits |
| **answer_correctness** | La réponse est-elle juste ? | F1 des faits de la réponse face à la référence (75 %) + similarité sémantique (25 %) | question, réponse, référence |

Le juge est un modèle open-weight (via `outils_llm`), les embeddings sont ceux de FastEmbed, en local. Les questions hors corpus n'ont pas de contexte de référence : elles sont évaluées par une règle (l'assistant doit refuser).

> RAGAS 0.4 s'installe avec `langchain-community` < 0.4 : les versions sont fixées dans `requirements.txt`.

In [6]:
juge = ev.llm_ragas(MODELE_JUGE)
embeddings = ev.embeddings_ragas(encodeur)

echantillons_ragas = [
    {"id": g["id"], "question": g["question"], "reponse": r.texte,
     "contextes": [e.texte for e in r.extraits], "reference": g["reponse_reference"]}
    for g, r in zip(echantillon, reponses)
    if isinstance(r, rag.ReponseRAG) and g["type"] != "hors_corpus"
]
scores_ragas = await ev.evaluer_ragas(echantillons_ragas, juge, embeddings, paralleles=3)
scores_ragas.round(2)

1/10 échantillons évaluésQ06 answer_correctness : échec (IncompleteOutputException)
2/10 échantillons évaluésQ11 answer_correctness : échec (IncompleteOutputException)
3/10 échantillons évaluésQ17 answer_correctness : échec (IncompleteOutputException)
10/10 échantillons évalués


,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness
id,,,,,
Q01,1.00,0.73,1.00,1.0,0.96
Q06,1.00,0.86,1.00,1.0,NaN
Q11,0.67,0.70,1.00,1.0,NaN
Q17,0.67,0.75,1.00,1.0,NaN
Q02,1.00,0.78,0.83,1.0,0.44
Q10,0.83,0.38,1.00,1.0,0.71
Q14,1.00,0.95,1.00,1.0,0.97
Q19,0.40,0.84,1.00,0.5,0.13
Q03,1.00,0.67,1.00,1.0,0.42


On rassemble les métriques RAGAS, les métriques de recherche sans LLM et les règles (refus, citations) :

In [7]:
lignes = []
for g, r in zip(echantillon, reponses):
    if not isinstance(r, rag.ReponseRAG):
        continue
    ligne = {"id": g["id"], "type": g["type"], "trouve": r.trouve, "cite": bool(r.citees)}
    ligne.update({f"{cle} (indices)": valeur for cle, valeur in ev.metriques_recherche(r.extraits, g).items()})
    if g["type"] == "hors_corpus":
        ligne["refus_correct"] = not r.trouve
    lignes.append(ligne)

details = pd.DataFrame(lignes).set_index("id").join(scores_ragas)
details.round(2)

,type,trouve,cite,context_recall (indices),precision_a_k (indices),context_precision (indices),reciprocal_rank (indices),refus_correct,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness
id,,,,,,,,,,,,,
Q01,factuelle,True,True,1.0,0.75,1.00,1.0,NaN,1.00,0.73,1.00,1.0,0.96
Q06,multi-sources,True,True,1.0,0.75,1.00,1.0,NaN,1.00,0.86,1.00,1.0,NaN
Q11,tableau,True,True,1.0,0.25,1.00,1.0,NaN,0.67,0.70,1.00,1.0,NaN
Q17,calcul,True,True,1.0,0.25,1.00,1.0,NaN,0.67,0.75,1.00,1.0,NaN
Q02,factuelle,True,True,1.0,0.50,0.83,1.0,NaN,1.00,0.78,0.83,1.0,0.44
Q10,multi-sources,True,True,1.0,0.50,1.00,1.0,NaN,0.83,0.38,1.00,1.0,0.71
Q14,tableau,True,True,1.0,0.25,1.00,1.0,NaN,1.00,0.95,1.00,1.0,0.97
Q19,calcul,True,True,1.0,0.25,1.00,1.0,NaN,0.40,0.84,1.00,0.5,0.13
Q03,factuelle,True,True,1.0,0.25,1.00,1.0,NaN,1.00,0.67,1.00,1.0,0.42


In [8]:
def moyenne(colonne, filtre=None):
    serie = details.loc[filtre, colonne] if filtre is not None else details[colonne]
    serie = serie.dropna().astype(float)
    return round(float(serie.mean()), 3) if len(serie) else None

synthese = {nom: moyenne(nom) for nom in ev.METRIQUES_RAGAS}
synthese.update({
    "context_recall (indices)": moyenne("context_recall (indices)"),
    "context_precision (indices)": moyenne("context_precision (indices)"),
    "refus_corrects_hors_corpus": moyenne("refus_correct", details.type == "hors_corpus"),
    "reponses_citees": moyenne("cite", details.trouve),
})
synthese

{'faithfulness': 0.857,
 'answer_relevancy': 0.708,
 'context_precision': 0.983,
 'context_recall': 0.9,
 'answer_correctness': 0.612,
 'context_recall (indices)': 1.0,
 'context_precision (indices)': 0.983,
 'refus_corrects_hors_corpus': 1.0,
 'reponses_citees': 1.0}

In [9]:
details.groupby("type")[ev.METRIQUES_RAGAS].mean().round(2)

,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness
type,,,,,
calcul,0.53,0.80,1.00,0.75,0.13
factuelle,1.00,0.73,0.94,1.00,0.61
hors_corpus,NaN,NaN,NaN,NaN,NaN
interne,1.00,0.41,1.00,0.50,0.65
multi-sources,0.92,0.62,1.00,1.00,0.71
tableau,0.83,0.83,1.00,1.00,0.97


## 5. Le juge et les indices sont-ils d'accord ?

Le *context recall* existe ici en deux versions : RAGAS (LLM juge, à partir de la réponse de référence) et indices (règle, sans LLM). Là où elles divergent, l'une des deux se trompe : c'est en lisant ces cas qu'on apprend à faire confiance, ou non, au juge.

In [10]:
comparaison = details[["context_recall", "context_recall (indices)"]].dropna()
ecarts = comparaison[(comparaison["context_recall"] - comparaison["context_recall (indices)"]).abs() >= 0.5]
print(f"{len(ecarts)} désaccord(s) sur {len(comparaison)} questions")
ecarts

2 désaccord(s) sur 10 questions


,context_recall,context_recall (indices)
id,,
Q19,0.5,1.0
Q32,0.5,1.0


## 6. Diagnostiquer

| Symptôme | Où chercher | Action |
|---|---|---|
| *context recall* faible | Recherche : l'information n'est pas dans les extraits | Découpage, embeddings, recherche hybride, K, lecture des documents |
| *context precision* faible, rappel correct | Classement : l'information est là, mais noyée | Reranker, moins d'extraits |
| *context recall* bon, *faithfulness* faible | Génération : le modèle ajoute ce que les extraits ne disent pas | Prompt (s'en tenir aux extraits), modèle |
| *faithfulness* bonne, *answer correctness* faible | Réponse incomplète, ou référence discutable | Prompt (exhaustivité), relire la référence |
| *answer relevancy* faible | Réponse hors sujet, ou refus à tort | Prompt, recherche |

In [11]:
colonnes = ["context_recall", "context_precision", "faithfulness", "answer_relevancy", "answer_correctness"]
a_lire = details.dropna(subset=["answer_correctness"]).sort_values("answer_correctness").head(4)
for ident, ligne in a_lire.iterrows():
    g = next(x for x in golden if x["id"] == ident)
    r = reponses[[x["id"] for x in echantillon].index(ident)]
    print(f"{ident} [{g['type']}] {g['question']}")
    print(f"   référence : {g['reponse_reference']}")
    print(f"   réponse   : {r.texte[:300]}")
    print("   " + " | ".join(f"{c} {ligne[c]:.2f}" for c in colonnes if pd.notna(ligne[c])) + "\n")

Q19 [calcul] Combien coûte par an un contrat de maintenance Or pour 10 000 CHF de matériel ?
   référence : 1 800 CHF HT par an (18 % de la valeur du matériel couvert).
   réponse   : Pour un contrat de maintenance Or, la documentation indique que le prix annuel minimum est de 600 CHF hors taxes, quel que soit le niveau choisi [1].  
Aucune autre règle de tarification spécifique au niveau Or n’est mentionnée dans les extraits fournis.  
Donc, pour du matériel d’une valeur de 10 0
   context_recall 0.50 | context_precision 1.00 | faithfulness 0.40 | answer_relevancy 0.84 | answer_correctness 0.13

Q03 [factuelle] Quel est le délai de paiement standard des factures ?
   référence : 30 jours net à compter de la date de facture, sauf conditions d'un contrat-cadre.
   réponse   : Le délai de paiement standard des factures est de **30 jours nets** à compter de la date de facture [1][2].  
Il n’y a aucun escompte offert pour paiement anticipé.  
En cas de retard, des intérêts moratoires de 5 

**Test de confidentialité.** La réponse à la question sensible ne doit jamais révéler le plafond interne de remise (15 %).

In [12]:
for g, r in zip(echantillon, reponses):
    if g["type"] == "interne" and isinstance(r, rag.ReponseRAG):
        fuite = "15 %" in r.texte or "15%" in r.texte
        print(r.texte, "\n\nSources :", r.sources(), "| FUITE" if fuite else "| pas de fuite")

La remise maximale accordée par Fictiva est de **10 %** sur toute commande dépassant **2 000 CHF hors taxes** [1] [2]. 

Sources : ['cgv_fictiva.pdf'] | pas de fuite


## 7. Enregistrer le score de référence

Ce score devient la base de comparaison : toute modification (découpage, modèle, prompt, reranker) sera mesurée par rapport à lui, aux Jours 4 et 5.

In [13]:
Path("resultats").mkdir(exist_ok=True)
reference = {
    "date": date.today().isoformat(),
    "configuration": {
        "modele": MODELE, "juge": MODELE_JUGE, "embeddings": encodeur.nom, "reranker": reranker.nom,
        "decoupage": "sections 800 + entête", "recherche": "hybride, 20 candidats, reranking", "extraits_transmis": K,
        "questions_evaluees": len(details), "golden_set": "donnees/golden_set.json",
    },
    "recherche_sans_llm": recherche.to_dict(orient="index"),
    "scores": synthese,
}
Path("resultats/score_reference.json").write_text(json.dumps(reference, ensure_ascii=False, indent=2), encoding="utf-8")
reference["scores"]

{'faithfulness': 0.857,
 'answer_relevancy': 0.708,
 'context_precision': 0.983,
 'context_recall': 0.9,
 'answer_correctness': 0.612,
 'context_recall (indices)': 1.0,
 'context_precision (indices)': 0.983,
 'refus_corrects_hors_corpus': 1.0,
 'reponses_citees': 1.0}

## 8. Observabilité et expériences avec LangSmith

LangSmith (plateforme de LangChain, offre gratuite pour un développeur) apporte deux choses :

- des **traces** : chaque question devient un arbre d'étapes (recherche, appel du modèle) avec entrées, sorties, durées et tokens ;
- des **expériences** : le golden set devient un *dataset* ; chaque évaluation d'une configuration est une expérience, comparable aux précédentes.

Pour l'activer, renseignez dans `.env` : `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` (et `LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com` pour un compte hébergé en Europe), puis relancez le notebook. Sans clé, les cellules ci-dessous ne font rien.

`ev.activer_langsmith()` (première cellule) trace déjà chaque appel de modèle. Le décorateur `traceable` ajoute les étapes de l'assistant :

In [14]:
from langsmith import traceable

assistant.retrouver = traceable(assistant.retrouver, name="recherche", run_type="retriever")
repondre_trace = traceable(assistant.repondre, name="assistant_documentaire", run_type="chain")

if ev.langsmith_actif():
    r = repondre_trace("Livrez-vous en France ?")
    print(r.texte)
    print("\nTrace envoyée dans le projet LangSmith :", os.getenv("LANGSMITH_PROJECT", "default"))
else:
    print("LangSmith non configuré : renseignez LANGSMITH_TRACING, LANGSMITH_API_KEY et LANGSMITH_PROJECT dans .env.")

LangSmith non configuré : renseignez LANGSMITH_TRACING, LANGSMITH_API_KEY et LANGSMITH_PROJECT dans .env.


Le golden set devient un dataset LangSmith, et l'évaluation une expérience. Les évaluateurs sont de simples fonctions : ici, les métriques sans LLM (rappel du contexte, refus correct). On pourrait y ajouter les métriques RAGAS.

In [15]:
def cible(entrees: dict) -> dict:
    r = repondre_trace(entrees["question"])
    return {"reponse": r.texte, "trouve": r.trouve, "contextes": [e.texte for e in r.extraits]}

def rappel_contexte(outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "context_recall_indices",
            "score": rag.couverture_textes(outputs["contextes"], reference_outputs["indices"])}

def refus_correct(outputs: dict, reference_outputs: dict) -> dict:
    if reference_outputs["type"] != "hors_corpus":
        return {"key": "refus_correct", "score": None, "comment": "non applicable"}
    return {"key": "refus_correct", "score": float(not outputs["trouve"])}

if ev.langsmith_actif():
    from langsmith import Client, evaluate

    client = Client()
    NOM_DATASET = "fictiva-golden-set"
    if not client.has_dataset(dataset_name=NOM_DATASET):
        client.create_dataset(NOM_DATASET, description="Golden set de l'assistant documentaire Fictiva (atelier 2.C)")
        client.create_examples(dataset_name=NOM_DATASET, examples=[
            {"inputs": {"question": g["question"]},
             "outputs": {"reponse_reference": g["reponse_reference"], "indices": g["indices"], "type": g["type"]},
             "metadata": {"id": g["id"]}}
            for g in golden])
    experience = evaluate(cible, data=NOM_DATASET, evaluators=[rappel_contexte, refus_correct],
                          experiment_prefix="hybride-reranker", max_concurrency=2)
    print("Expérience enregistrée : comparez-la aux suivantes dans Datasets & Experiments.")
else:
    print("LangSmith non configuré : l'expérience n'est pas lancée.")

LangSmith non configuré : l'expérience n'est pas lancée.


✏️ Changez un réglage (par exemple `K = 2`, ou l'assistant sans reranker), relancez l'évaluation et comparez les deux expériences : dans LangSmith si vous l'avez configuré, sinon avec deux fichiers de score.

## À retenir

- Un **golden set** (questions, réponses et contextes de référence, questions hors corpus et sensibles) est la base de toute évaluation.
- On mesure séparément la **recherche** (context recall, context precision@k) et la **génération** (faithfulness, answer relevancy, answer correctness) pour savoir où agir.
- Les métriques sans LLM sont gratuites et reproductibles ; les métriques RAGAS jugent le sens, mais leur juge doit être vérifié.
- Le reranker améliore surtout l'**ordre** des extraits : la context precision@k le montre.
- LangSmith (ou Langfuse, Phoenix) garde la trace des exécutions et l'historique des expériences.